<a href="https://colab.research.google.com/github/v-yunusic/REPO-PYTHON-VY/blob/main/transacciones-surtidor-2022/bencinera_2022.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QUEHACERES
* Extraer informacion a partir de columnas de tiempo (fecha inicio y termino)
* Encargarse de columna  ***duracion_carga_seg*** (30% datos vacios)
* Encargarse de valores atipicos (valores altos)

# CONTEXTO

El conjunto pertenece a datos registrados por nodos IoT durante los mes de enero a marzo 2022.

Se trata de una serie temporal y contiene en general la siguiente información:

  * Nro de la transacción.
  * Fecha y hora de inicio y de fin de la transacción.
  * Cantidad de litros de combustible despachado.
  * Volumen real registrado por otros nodos IoT en los tanques durante el inicio y fin de la descarga.
  * Estación, surtidor, empresa de origen.
  * Tipo de combustible transaccionado.
  * Geolocalización (si es soportada).
  * Unidad de consumo de destino.
  * Sector de la industria a la que pertenece la estación.

Desde la serie más atómica a la agrupación más general, el orden es el siguiente:
id_bomba --> id_equipo --> id_empresa --> id_industria

# PIP

In [ ]:
pip install matplotlib

In [ ]:
pip install kagglehub

In [ ]:
pip install rapidfuzz

# LIBRERIAS

In [ ]:
try:
  import numpy as np
  import matplotlib.pyplot as plt
  from scipy.stats import skew
  import pandas as pd
  import seaborn as sns
  import re # RegEx
  from sklearn.preprocessing import StandardScaler
  from sklearn.cluster import KMeans
  import gc

  #========= KAGGLE =========#
  import kagglehub

  #======= Rapidfuzz ========#
  from rapidfuzz.distance import Levenshtein

  print('Librerias importadas!')
except Exception as e:
  print(f'Librerias no importadas:\n{e}')

# PREPROCESAMIENTO

In [ ]:
# PETICION DE DATA HACIA KAGGLE

path = kagglehub.dataset_download("danielabosch/fuel-transactions-from-gas-pump")
print('DATASET DE KAGGLE CARGADO!')
print(path)

In [ ]:
data_path = path+'/transacciones_surtidor.csv'

In [ ]:
df = pd.read_csv(data_path, sep=',')
df.head()

In [ ]:
df.info()

In [ ]:
print(f'Cantidad de filas:\n{df.shape[0]}')
print(f'Cantidad de columnas:\n{df.shape[1]}')

In [ ]:
# DEFINICION DE COLUMNAS NUMERICAS Y CATEGORICAS
col_categoricas = df.select_dtypes(include=['object']).columns.tolist()
col_numericas = df.select_dtypes(exclude=['object','bool']).columns.tolist()
col_bools = df.select_dtypes(include=['bool']).columns.tolist()

In [ ]:
nulos = pd.DataFrame({
    "nulos": df.isna().sum(),
    "pct": df.isna().mean() * 100,
}).sort_values("pct", ascending=False)

nulos

In [ ]:
df.columns

In [ ]:
df = df[[
    'unique_id_transa',
    'cantidad',
    'cantidad_comp_15',
    'codigo_error',
    'timestamp',
    'timestamp_stop',
    'nombre_prod',
    'id_industria',
    'industria'
]]

In [ ]:
df.head(2)

In [ ]:
df.loc[df.duplicated()].shape[0]

In [ ]:
df['industria'] = df['industria'].fillna('Unknown')
df['nombre_prod'] = df['nombre_prod'].fillna('Unknown')

In [ ]:
df[['id_industria','industria']].loc[(df['id_industria'] == 0) & (df['industria']!='Unknown')].shape[0]

In [ ]:
df['id_industria'] = df['id_industria'].fillna(0)
df['id_industria'].unique()

In [ ]:
df = df.drop(columns='id_industria')

In [ ]:
df['industria'].unique()

In [ ]:
df.loc[pd.to_datetime(df['timestamp'], errors='coerce').isna()].shape[0], df.loc[pd.to_datetime(df['timestamp_stop'], errors='coerce').isna()].shape[0]

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df['timestamp_stop'] = pd.to_datetime(df['timestamp_stop'], errors='coerce')

In [ ]:
df.head(2)

# CLASIFICACION POR TIPO COMBUSTIBLE

In [ ]:
df.loc[df['nombre_prod'].isna()].shape[0]

In [ ]:
len(df['nombre_prod'].unique())

In [ ]:
df['nombre_prod'].unique()

In [ ]:
df['nombre_prod'] = df['nombre_prod'].str.rstrip().str.lstrip().str.upper()
df['nombre_prod'] = df['nombre_prod'].replace(r'[^a-zA-Z0-9áéíóúÁÉÍÓÚñÑ\s]', '', regex=True)

In [ ]:
df['tipo_producto'] = None
df.head(2)

In [ ]:
tipo_diesel = ['infinia','diesel','diésel','euro',
               'combustible','gasoil','gas oil','gom',
               'gas-oil','ultra','formula', 'ud']

tipo_nafta = ['nafta','super']

tipo_urea = ['urea','azul']

# Join une cada valor de las listas con el operando '|' (or)
df.loc[df['nombre_prod'].str.contains('|'.join(tipo_diesel), case=False), 'tipo_producto'] = 'Diesel'
df.loc[df['nombre_prod'].str.contains('|'.join(tipo_nafta), case=False), 'tipo_producto'] = 'Nafta'
df.loc[df['nombre_prod'].str.contains('|'.join(tipo_urea), case=False), 'tipo_producto'] = 'Urea'
df['tipo_producto'] = df['tipo_producto'].fillna('Otro')

In [ ]:
df.head(3)

In [ ]:
df['nombre_prod'].unique().shape[0]

In [ ]:
# MODELO BASE PARA DICCIONARIO DE PRODUCTOS
similares = pd.DataFrame(columns=['Producto','Similar','Similitud'])

similares['Producto'] = [p for p in df['nombre_prod'].unique() if p]
lista_similar = []
lista_scores = []

for p in similares['Producto']:
  if isinstance(p, str):
    producto = p

    result = [(Levenshtein.normalized_similarity(producto, str(p_similar)), str(p_similar))
              for p_similar in similares['Producto']
              if (Levenshtein.normalized_similarity(producto, str(p_similar)) != 1)]

    score, similar = max(result, key=lambda x : x[0])

    lista_scores.append(score)
    lista_similar.append(similar)

similares['Similar'] = lista_similar
similares['Similitud'] = lista_scores

In [ ]:
dict(zip(similares.Producto, similares.Similar))

In [ ]:
#ARREGLO MANUAL A PARTIR DE SIMILARES
dict_similares = {'PRODUCTO 2  DIESEL' : 'DIESEL',
 'PRODUCT 1': 'PRODUCTO 1',
 'PRODUCTO 1 EURO DIESEL': 'EURODIESEL',
 'EURO DIESEL GRADO3': 'EURODIESEL G3',
 'EURO G3': 'EURODIESEL G3',
 'VP NAFTA': 'VPOWER NAFTA',
 'SUPER': 'NAFTA SUPER',
 'GASOIL GOM G3 DIESEL': 'GASOIL GOM G3',
 'VP DIESEL': 'VPOWER DIESEL',
 'VPDIESEL': 'VPOWER DIESEL',
 'GAS OIL GRADO 2': 'GASOIL GRADO 2',
 'GAS OIL': 'GASOIL',
 'GASOIL 1': 'GASOIL',
 'GASOIL COMUN': 'GASOIL',
 'GASOIL GOM': 'GOM',
 'GASOIL  GRADO 3': 'GASOIL GRADO 3',
 'PRODUCT 2': 'PRODUCTO 2',
 'FORMULA': 'FORMULA DIESEL',
 'EURO DIESEL': 'EURODIESEL',
 'DIESEL500': 'DIESEL 500',
 'COMBUSTIBLE   DIESEL 500': 'DIESEL 500',
 'DIESEL COMUN': 'DIESEL',
 'ULTRADIESEL': 'ULTRA DIESEL',
 'GAS OIL 500': 'GASOIL 500',
 'INFINIA DIÉSEL': 'INFINIA DIESEL',
 'INFINIA': 'INFINIA DIESEL',
 'PRODUCTO 1  EURO DIESEL': 'EURODIESEL',
 'D 500': 'DIESEL 500',
 'GASOIL MARÍTIMO': 'GOM',
 'EURO': 'EURODIESEL',
 'DIESEL EURO': 'EURODIESEL',
 'UD': 'ULTRA DIESEL',
 'CEREALES NORTE SA': 'UNKNOWN',
 'PUERTO SECO LA PUERTA SA': 'UNKNOWN'}

In [ ]:
df['nombre_prod'] = df['nombre_prod'].map(dict_similares).fillna(df['nombre_prod'])

In [ ]:
df['nombre_prod'].unique().shape[0]

In [ ]:
df.columns

In [ ]:
df['codigo_error'] = df['codigo_error'].str.upper()
df['codigo_error'].unique()

In [ ]:
df['duracion_carga_seg'] = (df['timestamp_stop'] - df['timestamp']).dt.total_seconds()

#Lunes=0, Domingo=6
df['dia_de_semana'] = df['timestamp'].dt.dayofweek

In [ ]:
dias = {
    0 : 'Lunes',
    1 : 'Martes',
    2 : 'Miercoles',
    3 : 'Jueves',
    4 : 'Viernes',
    5 : 'Sabado',
    6 : 'Domingo'
}

In [ ]:
df['dia_de_semana'] = df['dia_de_semana'].map(dias)

In [ ]:
df['dia_de_semana'].unique()

In [ ]:
df = df[['unique_id_transa', 'cantidad',
       'cantidad_comp_15', 'codigo_error',
       'nombre_prod', 'tipo_producto', 'industria',
       'duracion_carga_seg', 'dia_de_semana']]

In [ ]:
df.head(1)

# OUTLIERS

### CANTIDAD COMBUSTIBLE

In [ ]:
df[['cantidad','cantidad_comp_15']].describe()

In [ ]:
round(df['duracion_carga_seg'].describe())

##### VALORES NEGATIVOS

In [ ]:
negativos = df.loc[(df['cantidad'] < 0) | (df['cantidad_comp_15'] < 0)].copy()
negativos.shape[0]

In [ ]:
# TRANSACCIONES BAJO CODIGO RS SON DE REABASTECIMIENTO
negativos.head()

In [ ]:
df = df.loc[(df['cantidad'] >= 0) | (df['cantidad_comp_15'] >= 0)]
df[['cantidad','cantidad_comp_15']].describe()

In [ ]:
df.shape[0]

##### VALORES CERO

In [ ]:
df1 = df.copy()
df1.head(), df1.shape[0]

In [ ]:
#APLICAMOS UNA MASCARA BOOLEANA DE MANERA QUE SE HAGA EL FILTRO SIEMPRE Y CUANDO SE CUMPLAN LAS 3 CONDICIONES
fila_vacia = (df['cantidad'] == 0) & (df['cantidad_comp_15'] == 0) & (df['duracion_carga_seg'] == 0)

In [ ]:
# IDENTIFICAMOS LAS COLUMNAS QUE NO NOS APORTAN NINGUN VALOR
ceros = df.loc[fila_vacia].copy()
ceros.shape[0], f'({round(ceros.shape[0]/df.shape[0]*100,1)}%)'

In [ ]:
ceros.head()

In [ ]:
df = df.loc[~fila_vacia]
df.shape[0]

In [ ]:
# Calculamos el porcentaje de forma más eficiente usando la media de la condición booleana
porcentaje_cero = (df['duracion_carga_seg'] == 0).mean() * 100
f"{porcentaje_cero:.2f}%"

In [ ]:
df.loc[(df['duracion_carga_seg'] == 0)]

In [ ]:
round(df['duracion_carga_seg'].describe())

In [ ]:
round(df['duracion_carga_seg'].ffill().describe())

In [ ]:
df1['cantidad_comp_15'] = df['cantidad_comp_15'].where(df['cantidad_comp_15'] > 0, np.nan)
df1['duracion_carga_seg'] = df['duracion_carga_seg'].where(df['duracion_carga_seg'] > 0, np.nan)
df1 = df1[['cantidad','cantidad_comp_15','duracion_carga_seg']]
df1.head()

In [ ]:
df

In [ ]:
df['cantidad_comp_15'].describe()

In [ ]:
cantidades_ffill = df1['cantidad_comp_15'].ffill().dropna()
duraciones_ffill = df1[]

In [ ]:
df = df.loc[(df['cantidad'] > 0) & (df['cantidad_comp_15'] > 0) & (df['duracion_carga_seg'] > 0)]

In [ ]:
percentil_95 = np.percentile(df['cantidad_comp_15'], 95)
percentil_95

In [ ]:
condition = (df['cantidad'] != 0) | (df['cantidad_comp_15'] != 0)
df[['cantidad','cantidad_comp_15']].where(condition, pd.NA)

In [ ]:
df.head(15)

In [ ]:
#IA
prom_industrias = df.groupby('industria')['cantidad_comp_15'].mean().reset_index()
prom_industrias.columns = ['industria', 'promedio (litros)']
prom_industrias = prom_industrias.sort_values('promedio (litros)', ascending=False)
prom_industrias

In [ ]:
prom_producto = df.groupby('nombre_prod')['cantidad_comp_15'].mean().reset_index()
prom_producto.columns = ['Producto','Promedio (litros)']
prom_producto = prom_producto.sort_values('Promedio (litros)', ascending=False)
prom_producto

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
sns.barplot(prom_producto, y="Producto", x="Promedio (litros)", hue='Producto', ax=ax)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
sns.barplot(prom_industrias, y="industria", x="promedio (litros)", hue='industria', ax=ax)

In [ ]:
sns.histplot(df1['cantidad_comp_15'], bins=30)

# CODIGO VIEJO

In [ ]:
productos = [p.rstrip().lstrip().upper() for p in df['nombre_prod'].unique() if p]
productos = [re.sub(r'[^a-zA-Z0-9áéíóúÁÉÍÓÚñÑ\s]', '', p) for p in productos if p]

productos = list(set(productos))

combustibles = pd.DataFrame(columns=['Producto','Tipo'])
combustibles['Producto'] = productos
combustibles.head(3)

In [ ]:
combustibles.shape[0]

In [ ]:
# TIPOS DE PRODUCTO:

tipo_diesel = ['infinia','diesel','diésel','euro',
               'combustible','gasoil','gas oil','gom',
               'gas-oil','ultra','formula', 'ud']

tipo_nafta = ['nafta','super']

tipo_urea = ['urea','azul']

for p in combustibles['Producto']:
  if isinstance(p, str):
    producto = p.lower()

    if any(t in producto for t in tipo_diesel):
      combustibles.loc[combustibles['Producto'].str.lower() == producto, 'Tipo'] = 'Diesel'

    if any(t in producto for t in tipo_nafta):
      combustibles.loc[combustibles['Producto'].str.lower() == producto, 'Tipo'] = 'Nafta'

    if any(t in producto for t in tipo_urea):
      combustibles.loc[combustibles['Producto'].str.lower() == producto, 'Tipo'] = 'Urea'

    combustibles['Tipo'] = combustibles['Tipo'].fillna('Otro')



In [ ]:
combustibles.loc[combustibles['Tipo'] == 'Diesel']

In [ ]:
combustibles['Tipo'].unique()